# 07_core16 — 민감도 A: Core 16개 변수 모형 (초안: core 담당 2026-11-26, 실행: logit 담당 ①, xgb 담당 ②③)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Walnutbrown/kdt14th-lendingclub/blob/main/notebooks/07_core16.ipynb)

선행 문헌의 Core 16개 원천 변수(`lib_prep.FEATURE_SPEC`에서 group = "core")만 써서 세 모형을 **Dev 전체로 한 번** 학습합니다. 부트스트랩은 하지 않고, Test에서 주 분석과 **같은 a\*(②의 a\*)**로 비교합니다(`05_test` 5절이 읽음).

- 하이퍼파라미터: 주 분석과 같은 `XGB_PARAMS` [미정·기본값: Core 16 전용 튜닝은 하지 않음 → 한계에 기재]
- 단조 부호: 주 분석 부호표에서 Core 변수 부분 그대로
- `DRY_RUN = True`: 부트스트랩 b=0의 학습 표본으로 학습(05_test 시험 실행과 짝)

In [ ]:
import sys
if "google.colab" in sys.modules:   # Colab: Drive 연결·레포 받기·notebooks/로 이동 (01_prep과 같은 방식)
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
import joblib
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

DRY_RUN = False
KINDS = ("logit",)   # 실행할 모형. ①(logit 담당)은 ("logit",), ②③(xgb 담당)은 ("xgb", "xgb_mono")로 바꿔 실행
X = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet")
if EXPECT.get("n_dev") and len(X) != EXPECT["n_dev"]:
    print(f"경고: Dev 행 수 {len(X):,} ≠ EXPECT {EXPECT['n_dev']:,}. parquet 버전 확인")
y = pd.read_parquet(DATA_DIR / "dev_meta.parquet")["y"].to_numpy()
core_cols = [out for _, (out, *_r, grp) in FEATURE_SPEC.items() if grp == "core"]
assert len(core_cols) == 16, core_cols
if DRY_RUN:   # model_template.make_oob_splits의 첫 회차와 같은 난수
    tr = np.random.default_rng(SEED).integers(0, len(X), size=len(X), dtype=np.int64)
    X, y = X.iloc[tr].reset_index(drop=True), y[tr]
OUT_C = OUT_DIR / "core16"; OUT_C.mkdir(parents=True, exist_ok=True)
CATS = load_categories()
print(X.shape, "Core:", core_cols)

In [ ]:
for kind in KINDS:
    prep = make_preprocessor(kind, CATS, mode="core_t0").fit(X[core_cols])
    Xt = prep.transform(X[core_cols])
    if kind == "logit":
        model = LogisticRegression(**LOGIT_PARAMS).fit(Xt, y)
    else:
        params = {**XGB_FIXED, **XGB_PARAMS}
        if kind == "xgb_mono":
            params["monotone_constraints"] = mono_constraints(prep)
        model = XGBClassifier(**params).fit(Xt, y)
    name = f"{'_dryrun_' if DRY_RUN else ''}{kind}.joblib"
    joblib.dump((prep, model), OUT_C / name)
    print(kind, Xt.shape[1], "열 →", (OUT_C / name).relative_to(ROOT))
(OUT_C / f"{'_dryrun_' if DRY_RUN else ''}summary.json").write_text(json.dumps(dict(
    core_cols=core_cols, kinds=list(KINDS), dry_run=DRY_RUN, config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH, meta=run_meta()),
    ensure_ascii=False, indent=1, default=str))